In [ ]:
import hashlib, json, shutil
from pathlib import Path
W = Path('/kaggle/working'); (W / 'tok').mkdir(exist_ok=True)
rep = {}
def sha(p): return hashlib.sha256(p.read_bytes()).hexdigest() if p.exists() else None
for cfg in sorted(Path('/kaggle/input').rglob('config.json')):
    d = cfg.parent
    if 'gemma-4-e4b' not in str(d): continue
    key = 'w4a16' if 'w4a16' in str(d) else 'bf16_unquantized'
    rep[key] = {'dir': str(d), 'files': sorted(p.name for p in d.iterdir())[:60]}
    for n in ('chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json', 'generation_config.json', 'special_tokens_map.json'):
        p = d / n
        rep[key][n + '_sha256'] = sha(p)
        if p.exists():
            (W / 'tok' / key).mkdir(parents=True, exist_ok=True)
            shutil.copy(p, W / 'tok' / key / n)
rep['identical_template'] = rep.get('w4a16', {}).get('chat_template.jinja_sha256') == rep.get('bf16_unquantized', {}).get('chat_template.jinja_sha256')
rep['identical_tokenizer'] = rep.get('w4a16', {}).get('tokenizer.json_sha256') == rep.get('bf16_unquantized', {}).get('tokenizer.json_sha256')
(W / 'tokenizer_check.json').write_text(json.dumps(rep, indent=1))
print(json.dumps(rep, indent=1))